<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · PROYECTO FINAL — MOTOR PREDICTIVO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Motor Predictivo de Éxito de Eventos Académicos</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Fine-tuning de BETO para regresión sobre reseñas académicas → Predicción de rating esperado</div></div>

## Objetivo

Entrenar un modelo de regresión sobre **BETO** (BERT en español, `dccuchile/bert-base-spanish-wwm-uncased`) usando el dataset `collegereview2021.csv` de Kaggle para predecir el **rating** (0–10) esperado de un evento académico a partir de su descripción textual.

El modelo entrenado se exporta como `modelo_predictivo.pt` y se integra en:
1. **DataGenerator** → inyectar evaluaciones sintéticas matemáticamente consistentes.
2. **nlp_service (FastAPI)** → endpoint `POST /api/ml/predict` para el Ponente en Flutter.

> ⚙️ **Entorno recomendado: Google Colab con GPU T4.** Entorno de ejecución → Cambiar tipo → **GPU**.

## 0 · Setup y GPU

In [ ]:
!pip install -q transformers datasets scikit-learn pandas numpy torch

import gc, json, math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Memoria liberada.')

---
## Parte A · Carga y Preprocesamiento del Dataset

**A.1** Carga de `collegereview2021.csv`. El dataset contiene reseñas de estudiantes universitarios con un `rating` del 0 al 10. Usaremos la columna `review` como entrada de texto y `rating` como variable objetivo (Y).

In [ ]:
# Subir el archivo desde tu computadora local
from google.colab import files
uploaded = files.upload()  # Selecciona collegereview2021.csv

df = pd.read_csv('collegereview2021.csv')
print(f'Shape original: {df.shape}')
print(df[['review', 'rating']].head(3))

**A.2** Limpieza y normalización. Eliminamos filas con NaN, truncamos reviews muy largas y normalizamos el rating al rango [0, 1] para facilitar el entrenamiento (regresión sigmoide).

$$\hat{y}_{norm} = \frac{rating}{10}$$

In [ ]:
# Limpiar
df = df[['review', 'rating']].dropna()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])
df = df[df['rating'].between(0, 10)]

# Normalizar rating a [0, 1]
df['rating_norm'] = df['rating'] / 10.0

# Truncar reviews a 512 caracteres (límite de BERT)
df['review'] = df['review'].astype(str).str[:512]

print(f'Registros limpios: {len(df)}')
print(f'Rating promedio: {df["rating"].mean():.2f}')
print(f'Distribución de ratings:\n{df["rating"].describe()}')

**A.3** División en conjuntos de entrenamiento y prueba (80/20), fijando `random_state=42` para reproducibilidad.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f'Entrenamiento: {len(train_df)} | Prueba: {len(test_df)}')

---
## Parte B · Tokenización con BETO

**B.1** Cargamos el tokenizer de **BETO** (`dccuchile/bert-base-spanish-wwm-uncased`), el equivalente español del BERT del Lab 6. Convierte cada review en una secuencia de `input_ids` y `attention_mask`.

In [ ]:
MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-uncased'
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

# Ejemplo de tokenización
ejemplo = train_df['review'].iloc[0]
tokens = tokenizer(ejemplo, truncation=True, max_length=256, padding='max_length', return_tensors='pt')
print(f'Texto: {ejemplo[:80]}...')
print(f'input_ids shape: {tokens["input_ids"].shape}')
print(f'Tokens decodificados (primeros 10): {tokenizer.convert_ids_to_tokens(tokens["input_ids"][0][:10])}')

**B.2** Dataset PyTorch personalizado. Encapsula la tokenización y devuelve tensores listos para el DataLoader.

In [ ]:
MAX_LEN = 256

class ReviewDataset(Dataset):
    def __init__(self, textos, ratings):
        self.textos = textos.tolist()
        self.ratings = ratings.tolist()

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.textos[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.ratings[idx], dtype=torch.float)
        }

BATCH_SIZE = 16
train_ds = ReviewDataset(train_df['review'], train_df['rating_norm'])
test_ds  = ReviewDataset(test_df['review'],  test_df['rating_norm'])
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'Batches de entrenamiento: {len(train_dl)}')
print(f'Batches de prueba:        {len(test_dl)}')

---
## Parte C · Arquitectura del Modelo de Regresión

**C.1** Definimos `BETORegressor`: usamos BETO como backbone encoder (igual que en el Lab 6), tomamos el token `[CLS]` y añadimos una **cabeza de regresión** (una capa lineal + sigmoide) para predecir el rating normalizado.

$$\hat{y} = \sigma(W \cdot h_{[CLS]} + b), \quad \hat{y} \in [0, 1]$$

$$\text{rating\_predicho} = \hat{y} \times 10$$

In [ ]:
class BETORegressor(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size  # 768 para BETO
        self.regressor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()  # Salida en [0, 1]
        )

    def forward(self, input_ids, attention_mask):
        # Extraer el vector [CLS] del último layer oculto
        salidas = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vector = salidas.last_hidden_state[:, 0, :]  # shape: (batch, 768)
        return self.regressor(cls_vector).squeeze(-1)    # shape: (batch,)

modelo = BETORegressor().to(DEVICE)
total_params = sum(p.numel() for p in modelo.parameters())
print(f'Parámetros totales: {total_params:,}')
print(f'Hidden size de BETO: {modelo.bert.config.hidden_size}')

---
## Parte D · Entrenamiento (Fine-tuning)

**D.1** Configuración del optimizador y función de pérdida. Usamos `AdamW` (igual que en el Lab 6) con learning rate diferenciado: bajo para BERT y alto para la cabeza de regresión. La pérdida es **MSE** (Error Cuadrático Medio):

$$\mathcal{L}_{MSE} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 3
LR_BERT = 2e-5
LR_HEAD = 1e-3

# Learning rate diferenciado: BERT se afina lento, la cabeza aprende rápido
optimizer = AdamW([
    {'params': modelo.bert.parameters(),       'lr': LR_BERT},
    {'params': modelo.regressor.parameters(),  'lr': LR_HEAD}
])

total_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
loss_fn = nn.MSELoss()

print(f'Total steps de entrenamiento: {total_steps}')
print(f'Warmup steps: {int(0.1 * total_steps)}')

**D.2** Loop de entrenamiento y validación.

In [ ]:
def epoch_train(modelo, dl, optimizer, scheduler, loss_fn):
    modelo.train()
    total_loss = 0
    for batch in dl:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        y     = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        pred = modelo(ids, mask)
        loss = loss_fn(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(dl)

def epoch_eval(modelo, dl, loss_fn):
    modelo.eval()
    total_loss, preds_all, y_all = 0, [], []
    with torch.no_grad():
        for batch in dl:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            y     = batch['label'].to(DEVICE)
            pred  = modelo(ids, mask)
            total_loss += loss_fn(pred, y).item()
            preds_all.extend(pred.cpu().numpy())
            y_all.extend(y.cpu().numpy())
    preds_all = np.array(preds_all) * 10   # Des-normalizar a escala 0-10
    y_all     = np.array(y_all)     * 10
    mae = mean_absolute_error(y_all, preds_all)
    r2  = r2_score(y_all, preds_all)
    return total_loss / len(dl), mae, r2

# --- Loop principal ---
print(f'{"Época":<6} | {"Loss Train":<12} | {"Loss Val":<10} | {"MAE (0-10)":<12} | R²')
print('-' * 60)
for epoch in range(1, EPOCHS + 1):
    loss_t = epoch_train(modelo, train_dl, optimizer, scheduler, loss_fn)
    loss_v, mae, r2 = epoch_eval(modelo, test_dl, loss_fn)
    print(f'{epoch:<6} | {loss_t:<12.4f} | {loss_v:<10.4f} | {mae:<12.4f} | {r2:.4f}')

---
## Parte E · Inferencia y Exportación del Modelo

**E.1** Función de inferencia. Recibe un texto libre (descripción de un evento) y devuelve el **rating esperado en escala 0–10** junto con el sentimiento predicho.

In [ ]:
def predecir_rating(texto: str) -> dict:
    """Predice el rating esperado (0-10) de un evento a partir de su descripción."""
    modelo.eval()
    encoding = tokenizer(
        texto,
        truncation=True,
        max_length=MAX_LEN,
        padding='max_length',
        return_tensors='pt'
    )
    with torch.no_grad():
        rating_norm = modelo(
            encoding['input_ids'].to(DEVICE),
            encoding['attention_mask'].to(DEVICE)
        ).item()

    rating = round(rating_norm * 10, 2)
    sentimiento = 'positivo' if rating >= 7 else ('neutral' if rating >= 4 else 'negativo')

    return {
        'rating_predicho': rating,
        'sentimiento':     sentimiento,
        'porcentaje_asistencia_estimado': round(rating * 10, 1)
    }

# Pruebas de inferencia
ejemplos = [
    "The faculty members were incredibly passionate and supportive. The library resources and research facilities were outstanding.",
    "Terrible experience. Overcrowded classrooms, unhelpful staff and very poor facilities. Would not recommend.",
    "Average college. Some professors are good, others not so much. The campus is decent."
]

print(f'{'Texto (primeros 60 chars)':<62} | Rating | Sentimiento')
print('-' * 90)
for ej in ejemplos:
    resultado = predecir_rating(ej)
    print(f'{ej[:60]:<62} | {resultado["rating_predicho"]:<6} | {resultado["sentimiento"]}')

**E.2** Guardar el modelo entrenado como `modelo_predictivo.pt` y descargarlo a tu computadora. Este archivo se sube a la carpeta `nlp_service/models_cache/` en el servidor EC2.

In [ ]:
import os

RUTA_MODELO = 'modelo_predictivo.pt'

torch.save({
    'model_state_dict': modelo.state_dict(),
    'model_name':       MODEL_NAME,
    'max_len':          MAX_LEN,
    'version':          '1.0.0'
}, RUTA_MODELO)

size_mb = os.path.getsize(RUTA_MODELO) / (1024 * 1024)
print(f'✅ Modelo guardado: {RUTA_MODELO} ({size_mb:.1f} MB)')

# Descargar desde Colab
from google.colab import files
files.download(RUTA_MODELO)

---
## Parte F · Integración Futura con el Proyecto

### Flujo de integración completo

```
┌─────────────────────────────────────────────────────────────┐
│  1. DataGenerator (Node.js / seed_massive.js)               │
│     ↓ POST /api/ml/predict  (texto del evento ficticio)     │
│  2. nlp_service (FastAPI / Python)                          │
│     ↓ carga modelo_predictivo.pt → devuelve rating         │
│  3. DataGenerator inyecta evaluación con rating REAL        │
│     (no random) en server_dev (PostgreSQL)                  │
│                                                             │
│  4. Flutter App → Pantalla Crear Evento del Ponente         │
│     ↓ Botón "🔮 Predecir Éxito"                            │
│     ↓ Node.js → Python → rating_predicho + sentimiento      │
│     ↓ Flutter muestra: "Tu evento tendría ★ 8.3 / 10"      │
└─────────────────────────────────────────────────────────────┘
```

### Pasos para desplegar el modelo en EC2
```bash
# 1. Desde tu computadora local, sube el archivo al servidor
scp -i NiggaFlex.pem modelo_predictivo.pt ubuntu@<IP_EC2>:~/INTEGER09-funx_de_ponente_id02/nlp_service/models_cache/

# 2. Reinicia el contenedor NLP para que cargue el nuevo modelo
docker compose restart nlp
```